## Import libs and mount drive



In [21]:
import os
import re
import sys
import json
import math
import uuid
import shutil
import select
import difflib
import hashlib
import zipfile
import datetime
import subprocess
import traceback
from collections import defaultdict

import numpy as np
import pandas as pd
import geopandas as gpd
import requests
import psycopg2
import gspread
from sqlalchemy import create_engine
from google.auth import default
from google.oauth2.service_account import Credentials

try:
    from pygbif import species
    from fuzzywuzzy import fuzz
except ImportError:
    subprocess.run(['pip', 'install', '-q', 'pygbif==0.6.5', 'fuzzywuzzy'], check=True)
    from pygbif import species
    from fuzzywuzzy import fuzz

# Load credentials from Colab Secrets (Colab) or environment variables (local)
try:
    from google.colab import drive, files, auth
    drive.mount('/content/drive')
    from google.colab import userdata
    _DB_URL       = userdata.get('IEN_DB_URL')
    _ZENODO_TOKEN = userdata.get('IEN_ZENODO_TOKEN')
except Exception:
    _DB_URL       = os.environ.get('IEN_DB_URL', '')
    _ZENODO_TOKEN = os.environ.get('IEN_ZENODO_TOKEN', '')

# Dataframe print options
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Defines

In [22]:
data_type = 'type_A'  ## Change between type_A and type_B

DRIVE_BASE    = '/content/drive/MyDrive/IEN Acoustics Dataset'
UPLOAD_BASE   = f'{DRIVE_BASE}/Acoustic Data Upload Form (File responses)'

TYPE_CONFIG = {
    'type_A': {
        'drive_folder':              f'{UPLOAD_BASE}/Upload Section A data (File responses)',
        'timestamp_cutoff':          datetime.datetime(2025, 2, 1, 0, 0),
        'mandatory_meta_cols':       ['Media File Name/Media URL', 'Location Latitude', 'Location Longitude', 'Recording Date', 'Authors', 'Annotation File Attached'],
        'mandatory_annotation_cols': ['Media File Name/Media URL', 'Begin Time (s)', 'End Time (s)', 'Low Freq (Hz)', 'High Freq (Hz)'],
        'mandatory_author_cols':     ['Author', 'Email ID', 'Affiliations'],
        'media_dest_folder':         f'{DRIVE_BASE}/typeA_processed_media',
        'xls_dest_folder':           f'{DRIVE_BASE}/typeA_processed_xls',
        'annotations_dest_folder':   f'{DRIVE_BASE}/typeA_annotations',
        'has_annotations':           True,
        'target_meta_cols':          ['Media File Name/Media URL', 'Location Latitude', 'Location Longitude', 'Recording Date', 'Authors',
                                      'Annotation File Attached', 'Hide Exact Location', 'Recording Type', 'Notes', 'annotation_found',
                                      'media_length_sec', 'bit_depth', 'sampling_rate', 'byte_size', 'binary_hash',
                                      'original_zip', 'new_media_path', 'new_xls_path', 'new_anno_path', 'state'],
    },
    'type_B': {
        'drive_folder':              f'{UPLOAD_BASE}/Upload Section B data (File responses)',
        'timestamp_cutoff':          datetime.datetime(2025, 6, 1, 0, 0),
        'mandatory_meta_cols':       ['Media File Name/Media URL', 'Location Latitude', 'Location Longitude', 'Recording Date', 'Authors', 'Scientific Name'],
        'mandatory_annotation_cols': [],
        'mandatory_author_cols':     ['Author', 'Email ID', 'Affiliations'],
        'media_dest_folder':         f'{DRIVE_BASE}/typeB_processed_media',
        'xls_dest_folder':           f'{DRIVE_BASE}/typeB_processed_xls',
        'annotations_dest_folder':   f'{DRIVE_BASE}/typeB_annotations',
        'has_annotations':           False,
        'target_meta_cols':          ['Media File Name/Media URL', 'Location Latitude', 'Location Longitude', 'Recording Date', 'Authors', 'Scientific Name',
                                      'Hide Exact Location', 'Notes', 'media_length_sec', 'bit_depth', 'sampling_rate', 'byte_size',
                                      'binary_hash', 'original_zip', 'new_media_path', 'new_xls_path', 'state', 'taxa_info'],
    },
}

# GBIF species lookup cache — persisted to Drive across runs
gbif_lookup_map  = defaultdict()
GBIF_info_map_path = f'{DRIVE_BASE}/GBIF_info_map.json'
try:
    with open(GBIF_info_map_path) as f:
        gbif_lookup_map = json.load(f)
except Exception:
    print('GBIF_info_map.json not found, starting fresh')

def persist_gbif_lookup_map():
    with open(GBIF_info_map_path, 'w') as f:
        json.dump(gbif_lookup_map, f)

class cfg:
    wait_for_inputs      = False  # If True, pause for user confirmation on fuzzy file matches
    media_col            = 'Media File Name/Media URL'
    accepted_media_types = ['.wav', '.flac', '.mp3', '.m4a', '.aiff', '.aac', '.ogg', '.wma']
    states_shape_file_path = f'{DRIVE_BASE}/resources/shapefile_valid.json'
    dataset_base_path    = f'{DRIVE_BASE}/zenodo_zips'
    db_url               = _DB_URL
    zenodo_token         = _ZENODO_TOKEN
    run_qc_sampling      = False  # Set True to execute the "Create QC sets" section

# Expand the selected data_type's config into cfg attributes
for _k, _v in TYPE_CONFIG[data_type].items():
    setattr(cfg, _k, _v)


## Meta and Annotation handling functions

In [ ]:
#!apt-get install -y ffmpeg

def fuzzy_match(query, choices):
    """Fuzzy match a query string against a list of choices. Returns best match or None."""
    choices_lower = [c.lower().strip() for c in choices]
    query_lower   = query.lower().strip()
    match_idx = difflib.get_close_matches(query_lower, choices_lower, n=1, cutoff=0.7)
    if match_idx:
        idx = choices_lower.index(match_idx[0])
        return choices[idx]
    return None


def annotations_standardize_fuzzy(df, standard_columns):
    """Rename submitted annotation columns to standard names via fuzzy matching."""
    col_map = {}
    for standard_col in cfg.mandatory_annotation_cols:
        match = fuzzy_match(standard_col, df.columns)
        if match:
            col_map[match] = standard_col
    print(col_map)
    return df.rename(columns=col_map)


def sanitize_filename(filename):
    """Replace characters invalid in Google Drive filenames with underscores."""
    sanitized = re.sub(r'[<>:"/\\|?*]', '_', filename)
    return sanitized.replace(' ', '_').replace("'", "")


def handle_media_file(df_row, zip_ref):
    unique_id  = str(uuid.uuid4())
    media_ref  = df_row[cfg.media_col].strip()
    authors    = df_row['Authors'].replace(' ', '').replace(',', '|')

    if isinstance(media_ref, str) and media_ref.lower().startswith(('http://', 'https://')):
        ext = os.path.splitext(media_ref)[1] or '.wav'
        unique_media_fname = sanitize_filename(f"{authors}_{os.path.splitext(os.path.basename(zip_path))[0]}_{unique_id}{ext}")
        media_dest_path    = os.path.join(cfg.media_dest_folder, unique_media_fname)
        try:
            response = requests.get(media_ref, stream=True)
            response.raise_for_status()
            with open(media_dest_path, 'wb') as f:
                for chunk in response.iter_content(chunk_size=8192):
                    f.write(chunk)
        except Exception as e:
            print(f"Failed to download media from URL {media_ref}: {e}")
            return None
    else:
        matching_files = [f for f in zip_ref.namelist()
                          if (os.path.basename(f) == media_ref
                              or os.path.splitext(os.path.basename(f))[0] == media_ref)
                          and os.path.splitext(os.path.basename(f))[1].lower() in cfg.accepted_media_types]
        if not matching_files:
            print(f"Media file {media_ref} not found in zip {zip_path}")
            close_matches = difflib.get_close_matches(
                media_ref,
                [os.path.basename(f) for f in zip_ref.namelist()
                 if os.path.splitext(os.path.basename(f))[1].lower() in cfg.accepted_media_types],
                n=1, cutoff=0.8)
            if close_matches:
                if cfg.wait_for_inputs:
                    is_close_match = input(f"Type yes if this is your file? {close_matches[0]}") or 'no'
                else:
                    is_close_match = 'yes'
                    print(f'WARN: Exact media name {media_ref} not found; Choosing close match: {close_matches[0]}')
                if is_close_match.lower() == 'yes':
                    matching_files = [f for f in zip_ref.namelist()
                                      if (os.path.basename(f) == close_matches[0]
                                          or os.path.splitext(os.path.basename(f))[0] == close_matches[0])
                                      and os.path.splitext(os.path.basename(f))[1].lower() in cfg.accepted_media_types]
                else:
                    return None
            else:
                return None
        media_in_zip_path  = matching_files[0]
        ext                = os.path.splitext(media_in_zip_path)[1]
        unique_media_fname = f"{authors}_{os.path.splitext(os.path.basename(zip_path))[0]}_{unique_id}{ext}"
        media_dest_path    = os.path.join(cfg.media_dest_folder, unique_media_fname)
        with zip_ref.open(media_in_zip_path) as src, open(media_dest_path, 'wb') as dst:
            dst.write(src.read())

    return media_dest_path


def generate_hash(row, bytes=8):
    """SHA-256 hash of a DataFrame row, truncated to `bytes` hex chars."""
    row_string = ':'.join(str(x) for x in row.values)
    return hashlib.sha256(row_string.encode('utf-8')).hexdigest()[-bytes:]


def handle_annotations(df_row, annotations_df, zip_ref):
    unique_id          = str(uuid.uuid4())
    anno_df            = pd.DataFrame()
    authors            = df_row['Authors'].replace(' ', '').replace(',', '|')
    fname_wo_ext       = os.path.splitext(df_row[cfg.media_col])[0]
    unique_anno_fname  = f"{authors}_{os.path.splitext(os.path.basename(fname_wo_ext))[0]}_{unique_id}.txt"
    annotation_dest_path = os.path.join(cfg.annotations_dest_folder, unique_anno_fname)

    if 'Annotation File Attached' in df_row.index and df_row['Annotation File Attached'].lower() == 'yes':
        annotation_file = [f for f in zip_ref.namelist()
                           if os.path.basename(f).startswith(fname_wo_ext)
                           and os.path.splitext(f)[1] in ['.csv', '.tsv', '.txt', '.rav']]
        if annotation_file:
            anno_file = annotation_file[0]
            anno_df   = pd.read_csv(zip_ref.open(anno_file), sep='\t', on_bad_lines='warn')
            anno_df[cfg.media_col] = df_row[cfg.media_col]
            with zip_ref.open(anno_file) as src, open(annotation_dest_path, 'wb') as dst:
                dst.write(src.read())
            annotation_found = True
        else:
            print(f"No annotation file found for {fname_wo_ext}")
            annotation_found = False
    else:
        anno_df      = annotations_df[annotations_df[cfg.media_col] == df_row[cfg.media_col]].copy()
        non_empty_rows = anno_df.all(axis=1).sum()
        if non_empty_rows > 0:
            annotation_found = True
            anno_df.to_csv(annotation_dest_path, sep='\t', index=False)
        else:
            annotation_found = False
            print(f"Missing rows in annotations sheet for {fname_wo_ext}")

    anno_df['hash'] = anno_df.apply(generate_hash, axis=1)
    return annotation_dest_path, annotation_found, anno_df


def extract_media_info_ffprobe(media_path):
    info = {'bit_depth': None, 'sampling_rate': None, 'byte_size': None,
            'media_length_sec': None, 'binary_hash': None}
    try:
        cmd = ['ffprobe', '-v', 'error', '-select_streams', 'a:0',
               '-show_entries', 'stream=sample_rate,bit_rate,bits_per_sample,duration',
               '-of', 'json', media_path]
        result = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
        data   = json.loads(result.stdout)
        stream = data['streams'][0] if data['streams'] else {}
        info['sampling_rate']    = int(stream['sample_rate'])    if 'sample_rate'    in stream else None
        info['bit_depth']        = int(stream['bits_per_sample']) if 'bits_per_sample' in stream else None
        info['media_length_sec'] = float(stream['duration'])     if 'duration'       in stream else None
        info['byte_size']        = os.path.getsize(media_path)
        hasher = hashlib.sha256()
        with open(media_path, 'rb') as f:
            for chunk in iter(lambda: f.read(8192), b''):
                hasher.update(chunk)
        info['binary_hash'] = hasher.hexdigest()
    except Exception as e:
        print(f"ffprobe error for {media_path}: {e}")
    return info


def default_annotations_df():
    return pd.DataFrame(columns=cfg.mandatory_annotation_cols)


## Iterate through drive

In [ ]:
def get_zip_files_after(folder, cutoff):
    zip_files = []
    zip_files_valid = []
    for fname in os.listdir(folder):
        if fname.endswith('.zip'):
            fpath = os.path.join(folder, fname)
            mtime = datetime.datetime.fromtimestamp(os.path.getmtime(fpath))
            if mtime > cutoff:
                zip_files.append(fpath)

    # pick manually edited zips if present
    for i, zip_path in enumerate(zip_files):
        manually_edited_zip_name = os.path.splitext(zip_path)[0] + '_createdbysameer' + os.path.splitext(zip_path)[1]
        if manually_edited_zip_name in zip_files or manually_edited_zip_name.split('/')[-1].startswith('IGNORE_'):
            #print(manually_edited_zip_name)
            pass # zip_files_valid.append(zip_path)
        else:
            zip_files_valid.append(zip_path)

    # dedupe the list
    zip_set = set(zip_files_valid)
    for i, z in enumerate(zip_set):
        print(f'[{i}] {z}')
    return list(zip_set)

zip_files = get_zip_files_after(drive_folder, timestamp_cutoff)

[0] /content/drive/MyDrive/IEN Acoustics Dataset/Acoustic Data Upload Form (File responses)/Upload Section A data (File responses)/Papadkhinidi & Tungareshwar Mar-Apr 02_TypeA - Nishant Bhagwat.zip
[1] /content/drive/MyDrive/IEN Acoustics Dataset/Acoustic Data Upload Form (File responses)/Upload Section A data (File responses)/Batwar_2_typeA_dataset - Pavithra Sundar_createdbysameer.zip
[2] /content/drive/MyDrive/IEN Acoustics Dataset/Acoustic Data Upload Form (File responses)/Upload Section A data (File responses)/Aroli_benchmark_TypeA - Mayuri Kotian_createdbysameer.zip
[3] /content/drive/MyDrive/IEN Acoustics Dataset/Acoustic Data Upload Form (File responses)/Upload Section A data (File responses)/Paul_Pop_files_for_submission_batch_1_createdbysameer.zip
[4] /content/drive/MyDrive/IEN Acoustics Dataset/Acoustic Data Upload Form (File responses)/Upload Section A data (File responses)/Chhichari_2_typeA_dataset - Pavithra Sundar_createdbysameer.zip
[5] /content/drive/MyDrive/IEN Acoust

## Process submissions one by one

In [ ]:
results                 = []
annotations_consolidated = []
authors_consolidated    = []
os.makedirs(cfg.media_dest_folder,       exist_ok=True)
os.makedirs(cfg.xls_dest_folder,         exist_ok=True)
os.makedirs(cfg.annotations_dest_folder, exist_ok=True)

for i, zip_path in enumerate(zip_files):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        print(zip_path)
        xls_files = [f for f in zip_ref.namelist() if f.endswith('.xls') or f.endswith('.xlsx')]
        for xls_file in xls_files:
            print(f'Processing meta xls: [{i}]{xls_file}')
            try:
                with zip_ref.open(xls_file) as file:
                    meta_df   = pd.read_excel(file, sheet_name='Meta')
                with zip_ref.open(xls_file) as file:
                    author_df = pd.read_excel(file, sheet_name='Authors')
            except Exception:
                print(f'Skipping xls: {xls_file} from zip {zip_path}')
                continue

            try:
                with zip_ref.open(xls_file) as file:
                    annotations_df = pd.read_excel(file, sheet_name='Annotations')
            except ValueError:
                annotations_df = default_annotations_df()

            try:
                meta_df.dropna(subset=[cfg.media_col], inplace=True)
                meta_df[['Location Longitude', 'Location Latitude']] =                     meta_df[['Location Longitude', 'Location Latitude']].fillna(value=0)
                meta_df.drop_duplicates(inplace=True)
                mandatory_filled = meta_df[cfg.mandatory_meta_cols].notnull().all(axis=1)
                missing_rows     = meta_df[~mandatory_filled]
                if not missing_rows.empty:
                    print(f"Missing mandatory columns in {zip_path}:")
                    continue
            except Exception as e:
                print(f"Error while processing {zip_path}: {str(e)}")
                continue

            xls_annotations = []
            for idx, row in meta_df.iterrows():
                media_dest_path = handle_media_file(row, zip_ref)
                print(row['Media File Name/Media URL'])
                if media_dest_path is None:
                    continue

                if cfg.has_annotations:
                    media_anno_df = pd.DataFrame()
                    annotation_dest_path, annotation_found, media_anno_df =                         handle_annotations(row, annotations_df, zip_ref)
                    missing_cols = [c for c in cfg.mandatory_annotation_cols if c not in media_anno_df.columns]
                    if missing_cols:
                        print(f"WARN: Missing/Non-Raven annotations for media {row[cfg.media_col]} in {zip_path}")
                        print(missing_cols)
                        continue
                    xls_annotations.append(media_anno_df)
                    meta_df.at[idx, 'annotation_found'] = annotation_found
                else:
                    annotation_dest_path = None
                    annotation_found     = False

                media_meta = extract_media_info_ffprobe(media_dest_path)
                for field in ['media_length_sec', 'bit_depth', 'sampling_rate', 'byte_size', 'binary_hash']:
                    meta_df.at[idx, field] = media_meta[field]
                meta_df.at[idx, 'original_zip'] = zip_path

                media_name_with_hash = sanitize_filename(
                    f"{row['Authors']}_{media_meta['binary_hash'][-8:]}_{row[cfg.media_col]}")
                media_path_with_hash = os.path.join(cfg.media_dest_folder, media_name_with_hash)
                os.rename(media_dest_path, media_path_with_hash)

                if annotation_found:
                    fname_wo_ext = os.path.splitext(media_name_with_hash)[0]
                    new_annotation_dest_path = os.path.join(cfg.annotations_dest_folder, fname_wo_ext + '.txt')
                    os.rename(annotation_dest_path, new_annotation_dest_path)
                    meta_df.at[idx, 'new_anno_path'] = new_annotation_dest_path
                meta_df.at[idx, 'new_media_path'] = media_path_with_hash

            if cfg.has_annotations and xls_annotations:
                xls_annotations_df = pd.concat(xls_annotations, ignore_index=True)

            unique_xls_fname = f"{os.path.splitext(os.path.basename(zip_path))[0]}_{os.path.basename(xls_file)}"
            xls_dest_path = os.path.join(cfg.xls_dest_folder, unique_xls_fname)
            with zip_ref.open(xls_file) as src, open(xls_dest_path, 'wb') as dst:
                dst.write(src.read())
            meta_df['new_xls_path'] = xls_dest_path

            try:
                if cfg.has_annotations:
                    annotations_consolidated.append(xls_annotations_df)
                results.append(meta_df)
                authors_consolidated.append(author_df)
            except ValueError:
                print("Empty meta Dataframe")

meta_all_df    = pd.concat(results, ignore_index=True)
authors_all_df = pd.concat(authors_consolidated, ignore_index=True)
if cfg.has_annotations:
    annotations_all_df = pd.concat(annotations_consolidated, ignore_index=True)
    print(len(annotations_all_df))


/content/drive/MyDrive/IEN Acoustics Dataset/Acoustic Data Upload Form (File responses)/Upload Section A data (File responses)/Amjhar_1_typeA_dataset_revised - Pavithra Sundar.zip
Processing meta xls: [0]Amjhar_1_typeA_dataset/amjhar_1.xlsx
AM_SD37_20200217_073000-2
AM_SD37_20200218_070000-2
AM_SD37_20200218_070000-5
AM_SD11_20200219_064000-3
AM_SD37_20200220_072000-1
AM_SD37_20200222_061500-5
AM1_SD34_20201227_062500-2
AM1_SD34_20201231_082000-4
AM2_SD38_20210301_075000-2
AM2_SD38_20210228_073000-5
AM2_SD38_20210219_082000-5
AM_SD11_20200220_073500-2
AM_SD11_20200220_073500-3
AM_SD11_20200220_073500-6
AM2_SD38_20210221_062500-2
AM2_SD38_20210223_082000-3
AM2_SD38_20210223_070000-4
AM2_SD38_20210301_083000-4
AM2_SD38_20210301_075000-6
AM2_SD38_20210301_083000-5
147




## Some more Utils

In [ ]:
def get_extension_with_ffprobe(file_path):
    """Use ffprobe to detect the actual container format; returns e.g. '.wav'."""
    try:
        cmd    = ['ffprobe', '-v', 'error', '-show_entries', 'format=format_name', '-of', 'json', file_path]
        result = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
        data   = json.loads(result.stdout)
        mapping = {'wav': '.wav', 'mp3': '.mp3', 'ogg': '.ogg', 'flac': '.flac'}
        primary = data['format']['format_name'].split(',')[0]
        return mapping.get(primary, f'.{primary}')
    except Exception:
        return None


def consolidte_fuzzy(df, target_column_name, additional_cols=[], threshold=60):
    """
    Consolidate messy column variants into one Series by fuzzy-matching column names.
    The exact target column takes priority; other matches are used as fallback via bfill.
    """
    relevant_cols = [target_column_name] if target_column_name in df.columns else []
    for col in df.columns:
        if col == target_column_name:
            continue
        if fuzz.token_sort_ratio(col.lower(), target_column_name.lower()) >= threshold or col in additional_cols:
            relevant_cols.append(col)
            print(f"'{col}' matched '{target_column_name}'")
    if not relevant_cols:
        print(f"No columns found matching '{target_column_name}' with threshold {threshold}.")
        return pd.Series(index=df.index, dtype=object)
    print(relevant_cols)
    return df[relevant_cols].bfill(axis=1).iloc[:, 0]


GBIF_FIELDS = [
    'matchType', 'usageKey', 'speciesKey', 'confidence',
    'kingdom', 'phylum', 'family', 'genus', 'class', 'rank', 'order',
    'kingdomKey', 'phylumKey', 'familyKey', 'genusKey', 'classKey', 'orderKey',
    'status', 'canonicalName', 'scientificName', 'acceptedUsageKey',
]

def create_taxa_info(result):
    return {k: result.get(k) for k in GBIF_FIELDS}


def get_taxonomic_info(row, add_common_name=False):
    """
    Look up scientific name via GBIF backbone. Resolves synonyms to accepted name.
    Returns (taxa_info_json, common_names_list, resolved_sci_name).
    resolved_sci_name equals the accepted canonical name when the submitted name is a synonym.
    """
    scientific_name  = row['sci_name']
    info             = {}
    final_names      = []
    resolved_sci_name = scientific_name

    if scientific_name in gbif_lookup_map:
        cached = gbif_lookup_map[scientific_name]
        return json.dumps(cached['info']), cached['final_names'], cached.get('resolved_sci_name', scientific_name)

    try:
        result = species.name_backbone(name=scientific_name)
        if result.get('status') == 'SYNONYM' and result.get('rank') == 'SPECIES':
            accepted_key    = result.get('acceptedUsageKey')
            result_accepted = species.name_usage(key=accepted_key)
            print(f"Synonym {result.get('canonicalName')} [{accepted_key}] -> {result_accepted.get('canonicalName')}")
            info = create_taxa_info(result_accepted)
            info.update({'submitted_sciname': scientific_name, 'matchType': 'SYNONYM',
                         'usageKey': accepted_key, 'acceptedUsageKey': accepted_key,
                         'status': result_accepted['taxonomicStatus']})
            resolved_sci_name = result_accepted.get('canonicalName', scientific_name)
        else:
            info = create_taxa_info(result)

        if result.get('matchType') == 'NONE' or 'usageKey' not in result:
            print(f"Could not resolve taxonomic backbone for: {scientific_name}")
        elif add_common_name:
            vernacular_data = species.name_usage(key=info['usageKey'], data='vernacularNames', limit=10)
            common_names    = {item['vernacularName'].lower()
                               for item in vernacular_data.get('results', [])
                               if item.get('language') == 'eng' and item.get('vernacularName')}
            final_names = [n.capitalize() for n in sorted(common_names)]

        if result.get('matchType') is not None:
            print(f'Add to GBIF map {scientific_name}')
            gbif_lookup_map[scientific_name] = {'info': info, 'final_names': final_names, 'resolved_sci_name': resolved_sci_name}

    except Exception as e:
        print(f"Error fetching data for {scientific_name}: {e}")

    finally:
        return json.dumps(info), final_names, resolved_sci_name


def join_statename(df, long_col_name='Location Longitude', lat_col_name='Location Latitude'):
    """Spatial join to attach Indian state name based on lat/long coordinates."""
    target_cols = df.columns.tolist()
    gdf    = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df[long_col_name], df[lat_col_name]))
    states = gpd.read_file(cfg.states_shape_file_path)
    gdf    = gdf.set_crs(states.crs)
    joined = gpd.sjoin(gdf, states, how='left', predicate='within')
    if 'state' in joined.columns:
        joined.drop(columns=['state'], inplace=True)
    else:
        target_cols.append('state')
    joined['state'] = joined['ST_NM']
    return joined[target_cols]


def annotations_cleanup(df):
    df.dropna(subset=['Begin Time (s)', 'End Time (s)'], inplace=True)
    df['sci_name'] = consolidte_fuzzy(df, 'Scientific Name', additional_cols=['Common name', 'Species'])


def meta_cleanup(df):
    df['Hide Exact Location'] = np.where(df['Hide Exact Location'] == 'Yes', 'Yes', 'No')
    df['Location Longitude']  = df['Location Longitude'].apply(lambda x: 0 if isinstance(x, str) else x)
    df['Location Latitude']   = df['Location Latitude'].apply(lambda x: 0 if isinstance(x, str) else x)


## Create final downloadable Type A and Type B master xls-es

In [ ]:
print(f'#Meta rows: {len(meta_all_df)}')
print(f'#Author rows: {len(authors_all_df)}')

target_anno_cols  = ['Begin Time (s)', 'End Time (s)', 'Low Freq (Hz)', 'High Freq (Hz)',
                     'Media File Name/Media URL', 'Organism Seen?', 'Notes', 'sci_name', 'taxa_info']
target_auth_cols  = ['Author', 'Email ID', 'Affiliations']

meta_cleanup(meta_all_df)
meta_all_df = join_statename(meta_all_df)

if cfg.has_annotations:
    print(f'#Annotation rows: {len(annotations_all_df)}')
    annotations_cleanup(annotations_all_df)
    print('Starting GBIF lookup')
    annotations_all_df[['taxa_info', 'common_names', 'sci_name']] = annotations_all_df.apply(
        get_taxonomic_info, axis=1, result_type='expand')
    persist_gbif_lookup_map()

    annotations_standardised_df = annotations_all_df[target_anno_cols]
    annotations_all_csv_name    = f'./annotations_all_{data_type}.csv'
    annotations_std_csv_name    = f'./annotations_standardised_{data_type}.csv'
    annotations_all_df.to_csv(annotations_all_csv_name, index=False)
    annotations_standardised_df.to_csv(annotations_std_csv_name, index=False)
    files.download(annotations_all_csv_name)
    files.download(annotations_std_csv_name)
else:
    meta_all_df['sci_name'] = meta_all_df['Scientific Name']
    print('Starting GBIF lookup for Type B')
    meta_all_df[['taxa_info', 'common_names', 'sci_name']] = meta_all_df.apply(
        get_taxonomic_info, axis=1, result_type='expand')

meta_std_df = meta_all_df[cfg.target_meta_cols].loc[:, lambda d: ~d.columns.duplicated()]

meta_all_csv_name = f'./meta_all_{data_type}.csv'
meta_std_csv_name = f'./meta_std_{data_type}.csv'
meta_all_df.to_csv(meta_all_csv_name, index=False)
meta_std_df.to_csv(meta_std_csv_name, index=False)
files.download(meta_all_csv_name)
files.download(meta_std_csv_name)

authors_all_csv_name = f'./authors_all_{data_type}.csv'
authors_all_df[target_auth_cols].to_csv(authors_all_csv_name, index=False)
files.download(authors_all_csv_name)


#Meta rows: 14650
#Author rows: 15
Starting GBIF lookup for Type B
Add to GBIF map Sitta castanea : INFO: {'matchType': 'EXACT', 'usageKey': 2484887, 'speciesKey': 2484887, 'confidence': 99, 'kingdom': 'Animalia', 'phylum': 'Chordata', 'family': 'Sittidae', 'genus': 'Sitta', 'class': 'Aves', 'rank': 'SPECIES', 'order': 'Passeriformes', 'kingdomKey': 1, 'phylumKey': 44, 'familyKey': 5283, 'genusKey': 2484878, 'classKey': 212, 'orderKey': 729, 'status': 'ACCEPTED', 'canonicalName': 'Sitta castanea', 'scientificName': 'Sitta castanea Lesson, 1830', 'acceptedUsageKey': None}, names: [] 
Add to GBIF map Anthus trivialis : INFO: {'matchType': 'EXACT', 'usageKey': 2490246, 'speciesKey': 2490246, 'confidence': 99, 'kingdom': 'Animalia', 'phylum': 'Chordata', 'family': 'Motacillidae', 'genus': 'Anthus', 'class': 'Aves', 'rank': 'SPECIES', 'order': 'Passeriformes', 'kingdomKey': 1, 'phylumKey': 44, 'familyKey': 5257, 'genusKey': 2490241, 'classKey': 212, 'orderKey': 729, 'status': 'ACCEPTED', 'c

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Create QC sets from preprocessed files

In [ ]:
if cfg.run_qc_sampling:
    processed_drive_folder = f'{DRIVE_BASE}/IEN_processed_data/08122025/'
    qc_drive_folder        = f'{DRIVE_BASE}/IEN QC/{data_type}'

    typeA_meta_f    = processed_drive_folder + 'meta_std_type_A_8dec.csv'
    typeA_anno_f    = processed_drive_folder + 'annotations_standardised_type_A_8dec.csv'
    typeA_authors_f = processed_drive_folder + 'authors_all_type_A_8dec.csv'
    typeB_meta_f    = processed_drive_folder + 'meta_std_type_B_8dec.csv'
    typeB_authors_f = processed_drive_folder + 'authors_all_type_B_8dec.csv'

    qc_seed    = 47
    sample_pc  = 0.02  # 2% of data sampled per state/class for QC
    qc_xls_columns = {
        'type_A': ['Media File Name/Media URL', 'Location Latitude', 'Location Longitude',
                   'Recording Date', 'Organism Seen?', 'Notes_anno', 'Notes'],
        'type_B': ['Media File Name/Media URL', 'Location Latitude', 'Location Longitude',
                   'Recording Date', 'Scientific Name', 'Notes', 'Reviewed OK', 'Rejection Reason'],
    }
    qc_master_columns = ['state', 'class', 'Media File Name/Media URL', 'qc_file_name']

    def create_qc_row(st='', cl='', m='', qcf=''):
        return pd.DataFrame([{'state': st, 'class': cl, 'Media File Name/Media URL': m, 'qc_file_name': qcf}])

    typea_qc_master_df  = pd.DataFrame(columns=qc_master_columns)
    typea_qc_master_xls = os.path.join(qc_drive_folder, data_type + '_QC_master.xlsx')

    if cfg.has_annotations:
        meta_df    = pd.read_csv(typeA_meta_f)
        anno_df    = pd.read_csv(typeA_anno_f)
        authors_df = pd.read_csv(typeA_authors_f)
    else:
        meta_df    = pd.read_csv(typeB_meta_f)
        authors_df = pd.read_csv(typeB_authors_f)

    meta_df = meta_df[meta_df['binary_hash'].notnull()].reset_index()

    if cfg.has_annotations:
        anno_df['class']  = anno_df['taxa_info'].apply(lambda x: json.loads(x).get('class'))
        meta_anno_df      = meta_df.join(anno_df.set_index('Media File Name/Media URL'),
                                         on='Media File Name/Media URL', how='left', rsuffix='_anno')
        state_stats = meta_anno_df.groupby(['state', 'class']).agg(
            distinct_media_count=('Media File Name/Media URL', 'nunique')).reset_index()
    else:
        meta_df['class'] = meta_df['taxa_info'].apply(lambda x: json.loads(x).get('class'))
        meta_anno_df     = meta_df
        state_stats      = meta_df.groupby(['state', 'class']).agg(
            distinct_media_count=('Media File Name/Media URL', 'nunique')).reset_index()

    print(state_stats.to_markdown(index=False))

    for _, row in state_stats.iterrows():
        if row['class'] is None:
            print(f"State {row['state']} QC folder to be created later")
            continue
        sampled_df = meta_anno_df[
            (meta_anno_df['state'] == row['state']) & (meta_anno_df['class'] == row['class'])
        ].sample(n=math.ceil(row['distinct_media_count'] * sample_pc), random_state=qc_seed).reset_index()
        sampled_df['Reviewed OK']      = ''
        sampled_df['Rejection Reason'] = ''
        for idx, row in sampled_df.iterrows():
            qc_folder    = os.path.join(qc_drive_folder, row['state'], row['class'])
            os.makedirs(qc_folder, exist_ok=True)
            qc_file_name = os.path.join(qc_folder, f'{idx+1:04d}_{row["binary_hash"][-8:]}')
            shutil.copy(row['new_media_path'], qc_file_name + os.path.splitext(row['new_media_path'])[1])
            if cfg.has_annotations:
                shutil.copy(row['new_anno_path'], qc_file_name + os.path.splitext(row['new_anno_path'])[1])
            qc_row = create_qc_row(row['state'], row['class'], row['Media File Name/Media URL'], qc_file_name)
            typea_qc_master_df = pd.concat([typea_qc_master_df, qc_row])
            print(row['state'], row['class'], qc_file_name)
        sampled_df[qc_xls_columns[data_type]].to_csv(
            os.path.join(qc_folder, f"{row['state']}_{row['class']}_meta.csv"), index=False)

    typea_qc_master_df.to_excel(typea_qc_master_xls, index=False)


'\n#!pip install openpyxl\nimport math\n\nprocessed_drive_folder = \'/content/drive/MyDrive/IEN Acoustics Dataset/IEN_processed_data/08122025/\'\nqc_drive_folder = \'/content/drive/MyDrive/IEN Acoustics Dataset/IEN QC/\'+data_type\n\ntypeA_meta_f = processed_drive_folder + \'meta_std_type_A_8dec.csv\'\ntypeA_anno_f = processed_drive_folder + \'annotations_standardised_type_A_8dec.csv\'\ntypeA_authors_f = processed_drive_folder + \'authors_all_type_A_8dec.csv\'\n\ntypeB_meta_f = processed_drive_folder + \'meta_std_type_B_8dec.csv\'\ntypeB_authors_f = processed_drive_folder + \'authors_all_type_B_8dec.csv\'\n\n# Comment this line out\ndata_type = \'type_A\'\n\nqc_seed = 47\nsample_pc = 0.02    # 5% of data to be sampled for QC\nqc_xls_columns = {\'type_A\' :[\'Media File Name/Media URL\', \'Location Latitude\', \'Location Longitude\',\n       \'Recording Date\', \'Organism Seen?\', \'Notes_anno\', \'Notes\'],\n                  \'type_B\' : [\'Media File Name/Media URL\', \'Location Lati

## Pick data from DB to process it further

In [ ]:
anno_query  = "SELECT * FROM ien.anno;"
typeb_query = "SELECT * FROM ien.typeb_meta;"
typea_query = "SELECT * FROM ien.typea_meta;"

try:
    conn = psycopg2.connect(cfg.db_url)
    try:
        df_anno    = pd.read_sql_query(anno_query,  conn)
        df_meta_b  = pd.read_sql_query(typeb_query, conn)
        df_meta_a  = pd.read_sql_query(typea_query, conn)
        print(len(df_anno), len(df_meta_b), len(df_meta_a))
    except Exception as e:
        print(f"Error executing query: {e}")
    finally:
        conn.close()
        print("Database connection closed.")
except Exception as e:
    print(f"Connection failed: {e}")


/tmp/ipython-input-6437/1026898654.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_anno = pd.read_sql_query(anno_query, conn)
/tmp/ipython-input-6437/1026898654.py:15: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_meta_b = pd.read_sql_query(typeb_query, conn)
/tmp/ipython-input-6437/1026898654.py:16: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_meta_a = pd.read_sql_query(typea_query, conn)


37445 27455 1710
Database connection closed.


## Insert back to DB

In [ ]:
# refresh gbif map if needed
# gbif_lookup_map = {}

df_anno[['taxa_info', 'common_names', 'sci_name']]   = df_anno.apply(get_taxonomic_info,   axis=1, result_type='expand')
df_meta_b[['taxa_info', 'common_names', 'sci_name']] = df_meta_b.apply(get_taxonomic_info, axis=1, result_type='expand')

persist_gbif_lookup_map()

engine = create_engine(cfg.db_url)
df_anno.to_sql(  'anno_25feb',       con=engine, schema='ien', if_exists='append', index=False)
df_meta_b.to_sql('typeb_meta_25feb', con=engine, schema='ien', if_exists='append', index=False)
df_meta_a.to_sql('typea_meta_25feb', con=engine, schema='ien', if_exists='append', index=False)


710

## Create final Zenodo submission zips

In [ ]:
# Pick 1 random sci_name for each media file
typea_query = "select distinct on (m.media_file_name) \
      'type_A' as data_type, \
      m.new_media_path as new_media_path, \
      lpad((row_number() over (partition by state order by m.media_file_name))::text,5, '0')||'_'||lower(left(state,1))||replace(right(state, length(state)-1),' ','')||'_'||replace(recording_date::text,'-','')||'_'||right(m.binary_hash, 8) as archive_file_name, \
      state \
      from ien.typea_meta m \
      left join ien.anno a \
      on m.id =a.media_id \
      where taxa_info->>'usageKey' is not null;"
typeb_query = "select distinct on (media_file_name) \
      'type_B' as data_type, \
      new_media_path, \
      lpad((row_number() over (partition by state order by media_file_name))::text,5, '0')||'_'||lower(left(state,1))||replace(right(state, length(state)-1),' ','')||'_'||replace(recording_date::text,'-','')||'_'||right(binary_hash, 8) as archive_file_name, \
      media_file_name, state from ien.typeb_meta;"
background_query = "select distinct on (m.media_file_name) \
      m.new_media_path as new_media_path, \
      'background' as data_type, \
      lpad((row_number() over (partition by state order by m.media_file_name))::text,5, '0')||'_'||lower(left(state,1))||replace(right(state, length(state)-1),' ','')||'_'||replace(recording_date::text,'-','')||'_'||right(m.binary_hash, 8) as archive_file_name, \
      state \
      from ien.typea_meta m \
      left join ien.anno a \
      on m.id =a.media_id \
      where taxa_info->>'usageKey' is null;"

#typea_meta_anno_query = "Select * from ien.typea_meta left join ien.anno on ien.typea_meta.id = ien.anno.media_id;"

try:
    conn = psycopg2.connect(cfg.db_url)
    if conn:
        try:
            df_meta_b = pd.read_sql_query(typeb_query, conn)
            df_meta_a = pd.read_sql_query(typea_query, conn)
            df_bg = pd.read_sql_query(background_query, conn)

            print(len(df_meta_b), len(df_meta_a), len(df_bg))
        except Exception as e:
            print(f"Error executing query or loading data: {e}")
        finally:
            # It's important to close the connection after the operation
            conn.close()
            print("Database connection closed.")

except Exception as e:
    print(f"Connection failed: {e}")


# Combine the dataframes for easier iteration
df_combined = pd.concat([df_meta_a, df_meta_b, df_bg], ignore_index=True)

# Metadata list to track our operations
log_data = []

# Get unique states to create archives for
type_states = (df_combined['data_type']+ '|' + df_combined['state']).unique()  # TODO: handle blank state

for dataType in ['type_A', 'type_B', 'background']:
    os.makedirs(os.path.join(cfg.dataset_base_path, dataType), exist_ok=True)


for type_state in type_states:
    dataType, state = type_state.split('|')
    print(dataType, state)
    if not state: continue # Skip if state is empty

    zip_name = f"audio_{dataType.replace('_','')}_{state.replace(' ', '_')}.zip"
    zip_path = os.path.join(cfg.dataset_base_path, dataType, zip_name)

    # Filter files belonging to this state
    type_state_df = df_combined[(df_combined['state'] == state) & (df_combined['data_type'] == dataType)]
    files_processed_count = 0

    with zipfile.ZipFile(zip_path, 'w') as zipf:
        for _, row in type_state_df.iterrows():
            source_path = row['new_media_path']

            if os.path.exists(source_path):
                # 1. Extract original extension
                ff_ext = get_extension_with_ffprobe(source_path)
                extension = ff_ext if ff_ext else ".bin" # Default to .bin if all fails

                # 2. Construct new filename
                # format: <sci_name>_<recording_date>_<binary_hash>.<ext>
                new_filename = row['archive_file_name']
                new_filename = new_filename.replace(" ", "_") # Safety for filesystems

                # 3. Add to zip with the new name
                zipf.write(source_path, arcname=new_filename)

                # 4. Log the individual file movement
                log_data.append({
                    "original_file_name": os.path.basename(source_path),
                    "new_name": new_filename,
                    "new_archive_name": zip_name,
                    "state": state
                })
                files_processed_count += 1
            else:
                print(f"Warning: File not found at {source_path}")

    # 5. Capture Zip Statistics
    if os.path.exists(zip_path):
        zip_size_mb = os.path.getsize(zip_path) / (1024 * 1024)
        print(f"Created {zip_name}: {files_processed_count} files, {zip_size_mb:.2f} MB")

# Create the summary DataFrame
df_log = pd.DataFrame(log_data)

# Create a summary of the archives created
archive_summary = df_log.groupby('new_archive_name').agg(
    number_of_media_files=('new_name', 'count')
).reset_index()

# Note: Zip size is calculated after the loop for the summary
archive_summary['zip_size_mb'] = archive_summary['new_archive_name'].apply(
    lambda x: os.path.getsize(os.path.join(cfg.dataset_base_path, x)) / (1024 * 1024) if os.path.exists(os.path.join(cfg.dataset_base_path, x)) else 0
)

print("\n--- Process Complete ---")
print(archive_summary)

/tmp/ipykernel_1012/3932265324.py:38: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_meta_b = pd.read_sql_query(typeb_query, conn)
/tmp/ipykernel_1012/3932265324.py:39: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_meta_a = pd.read_sql_query(typea_query, conn)
/tmp/ipykernel_1012/3932265324.py:40: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_bg = pd.read_sql_query(background_query, conn)


10073 1703 342
Database connection closed.
type_A Arunachal Pradesh
Created audio_typeA_Arunachal_Pradesh.zip: 11 files, 473.91 MB
type_A Bihar
Created audio_typeA_Bihar.zip: 12 files, 25.14 MB
type_A Kerala
Created audio_typeA_Kerala.zip: 167 files, 8830.88 MB
type_A Uttarakhand
Created audio_typeA_Uttarakhand.zip: 104 files, 1773.70 MB
type_A Karnataka
Created audio_typeA_Karnataka.zip: 342 files, 4907.26 MB
type_A Himachal Pradesh
Created audio_typeA_Himachal_Pradesh.zip: 206 files, 2198.67 MB
type_A Andhra Pradesh
Created audio_typeA_Andhra_Pradesh.zip: 6 files, 20.75 MB
type_A Tamil Nadu
Created audio_typeA_Tamil_Nadu.zip: 192 files, 2004.90 MB
type_A Assam
Created audio_typeA_Assam.zip: 22 files, 1243.44 MB
type_A Nagaland
Created audio_typeA_Nagaland.zip: 2 files, 23.96 MB
type_A Jharkhand
Created audio_typeA_Jharkhand.zip: 129 files, 353.69 MB
type_A Maharashtra
Created audio_typeA_Maharashtra.zip: 403 files, 2694.81 MB
type_A West Bengal
Created audio_typeA_West_Bengal.zip: 6 

## Create Zenodo Metadata and Annotation csvs

In [24]:
# Pick 1 random sci_name for each media file
typea_metadata_query = """select distinct on (m.media_file_name)
    m.info->>'dataset_file_name' as media_file_name,
	  'audio_type_A_'||state||'.zip' as archive_name,
	  recording_date,
	  case when hide_exact_location is true then 'NA' else longitude::text end as longitude,
	  case when hide_exact_location is true then 'NA' else latitude::text end as latitude,
	  replace(authors, ',', ';') as author_names,
	  m.info->>'email_ids' as author_emails,
	  m.notes,
	  jsonb_build_object('media_length_seconds', media_length_sec::numeric(8,1), 'bit_depth', bit_depth::int, 'sampling_rate_KHz', (sampling_rate/1000)::int, 'byte_size_Mb', (byte_size/(1024*1024))::numeric(8,1)) as media_info
      from ien.typea_meta m
      left join ien.anno a
      on m.id = a.media_id
      where taxa_info->>'usageKey' is :is_background null;"""

typeb_metadata_query = """select distinct on (m.media_file_name, m.sci_name)
    m.info->>'dataset_file_name' as media_file_name,
	  'audio_type_B_'||state||'.zip' as archive_name,
	  m.sci_name as scientific_name,
	  recording_date,
	  case when hide_exact_location is true then 'NA' else longitude::text end as longitude,
	  case when hide_exact_location is true then 'NA' else latitude::text end as latitude,
	  replace(authors, ',', ';') as author_names,
	  m.info->>'email_ids' as author_emails,
	  m.notes,
	  case when q.validation is true then '1' when q.validation is false then '0' else 'NA' end as validation,
	  validation_comments->>'rejection_classification' as validation_fail_reason,
	  jsonb_build_object('reviewer_comments', validation_comments->>'validation_comments' , 'comment_explanation', validation_comments->>'specific_rejetion_category') as validation_comments,
	  taxa_info,
	  jsonb_build_object('media_length_seconds', media_length_sec::numeric(8,1), 'bit_depth', bit_depth::int, 'sampling_rate_KHz', (sampling_rate/1000)::int, 'byte_size_Mb', (byte_size/(1024*1024))::numeric(8,1)) as media_info

      from ien.typeb_meta m
	  left join ien.typeb_qc q on q.media_file_name=m.media_file_name;"""


typea_annotations_query = """select m.info->>'dataset_file_name' as media_file_name,
	a.begin_time_s as begin_time,
	a.end_time_s as end_time,
	a.high_freq_hz as high_freq,
	a.low_freq_hz as low_freq,
	a.sci_name as scientific_name,
	case when q.validation is true then '1' when q.validation is false then '0' else 'NA' end as validation,
	validation_comments->>'rejection_classification' as validation_fail_reason,
	case when q.validation is null then null else jsonb_build_object('reviewer_comments', validation_comments->>'validation_comments' , 'comment_explanation', validation_comments->>'specific_rejetion_category') end as validation_comments,
	a.notes,
	taxa_info

	from ien.anno a
	left join ien.typea_meta m
	on m.id = a.media_id
	left join ien.typea_qc q
	on (q.media_file_name, q.begin_time_s, q.end_time_s, q.high_freq_hz, q.low_freq_hz, q.sci_name) = (a.media_file_name, a.begin_time_s, a.end_time_s, a.high_freq_hz, a.low_freq_hz, a.sci_name)
	where taxa_info->>'usageKey' is :is_background null;"""

#typea_meta_anno_query = "Select * from ien.typea_meta left join ien.anno on ien.typea_meta.id = ien.anno.media_id;"

try:
    conn = psycopg2.connect(cfg.db_url)
    if conn:
        try:
            df_meta_b = pd.read_sql_query(typeb_metadata_query, conn)
            df_meta_a = pd.read_sql_query(typea_metadata_query.replace(':is_background', 'not'), conn)
            df_meta_bg = pd.read_sql_query(typea_metadata_query.replace(':is_background', ''), conn)

            df_anno_a = pd.read_sql_query(typea_annotations_query.replace(':is_background', 'not'), conn)
            df_anno_bg = pd.read_sql_query(typea_annotations_query.replace(':is_background', ''), conn)

        except Exception as e:
            print(f"Error executing query or loading data: {e}")
        finally:
            # It's important to close the connection after the operation
            conn.close()
            print("Database connection closed.")

except Exception as e:
    print(f"Connection failed: {e}")

df_meta_a['author_emails'] = df_meta_a['author_emails'].str.replace('"', '').replace('[', '').replace(']', '').replace(',', ';')
df_meta_b['author_emails'] = df_meta_b['author_emails'].str.replace('"', '').replace('[', '').replace(']', '').replace(',', ';')
df_meta_bg['author_emails'] = df_meta_bg['author_emails'].str.replace('"', '').replace('[', '').replace(']', '').replace(',', ';')

df_anno_bg.drop(columns=['taxa_info'], inplace=True) # not relevant for background

df_meta_b.to_csv('type_B_metadata.csv',index=False)
df_meta_a.to_csv('type_A_metadata.csv',index=False)
df_meta_bg.to_csv('background_metadata.csv',index=False)

df_anno_a.to_csv('type_A_annotations.csv',index=False)
df_anno_bg.to_csv('background_annotations.csv',index=False)

for f in ['type_B_metadata.csv', 'type_A_metadata.csv', 'background_metadata.csv',\
          'type_A_annotations.csv', 'background_annotations.csv']:
          files.download(f)




/tmp/ipykernel_6769/134748450.py:62: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_meta_b = pd.read_sql_query(typeb_metadata_query, conn)
/tmp/ipykernel_6769/134748450.py:63: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_meta_a = pd.read_sql_query(typea_metadata_query.replace(':is_background', 'not'), conn)
/tmp/ipykernel_6769/134748450.py:64: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_meta_bg = pd.read_sql_query(typea_metadata_query.replace(':is_background', ''), conn)
/tmp/ipykernel_6769/134748450.py:66: UserW

Database connection closed.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Upload zips and Metadata to Zenodo

Record descriptuion and certain other details such as creator roles are updated manually in zenodo Web UI because its not supported via APIs


In [ ]:
# 0. Templates and Author list
AUTHOR_DETAILS_GSHEET_URL = 'https://docs.google.com/spreadsheets/d/1nGeyW9Y0bXgDRe4FQamC9F1Fsy-gq3aJ3YqzAWDspk4/edit?usp=sharing'

# Unused
author_list = [
'vr292@cornell.edu',
'sameersingh.mail@gmail.com',
'paulvpop@gmail.com',
'poojamchoksi@gmail.com',
'prabhjeet_s.sias19@krea.ac.in',
'sarika.khanwilkar@gmail.com',
'sonalteotia89@gmail.com',
'pralhad.burli@gmail.com',
'kadambari.devarajan@gmail.com'
]

zenodo_creator_list = [{'name': 'Singh, Sameer', 'affiliation': 'Independant', 'orcid': '0009-0000-7236-6042', 'type': 'DataManager'},
                       {'name': 'Ramesh, Vijay', 'affiliation': 'Cornell Lab of Ornithology', 'orcid': '0000-0002-0738-8808', 'type': 'ProjectLeader'},
                       {'name': 'Pop, Paul', 'affiliation': 'Ashoka Trust for Research in Ecology and the Environment', 'orcid': '0000-0003-0679-7789', 'type': 'DataCurator'}]

contributor_list = {'type_A':[
'priyankarchakraborty1991@gmail.com', 'sameersingh.mail@gmail.com', 'ramitsingal@gmail.com', 'ankita.dasgupta@darukaa.com', 'chandran@unknown.com',
'vishnu@keystone-foundation.org', 'anish@unknown.com', 'ashankar@tifrh.res.in', 'cswathi@tifrh.res.in','harshakumar@tifrh.res.in',
'kvd.novel@gmail.com', 'ssangeethspeaks@gmail.com', 'ryansatish22@gmail.com', 'komal.meena@darukaa.com', 'gaurav.singh@darukaa.com',
'paulvpop@gmail.com', 'yana@uni.minerva.edu', 'tasfiasyeda22@gmail.com', 'aavikadhanda@gmail.com','rajkamal@atree.org',
'baishyarohan16@gmail.com','dsouzasenan@gmail.com', 'malavikamadhavan98@gmail.com', 'kavya@ncf-india.org', 'amisha@ncf-india.org',
'elrika@ncf-india.org', 'rmathesan@keystone-foundation.org', 'navneet.singh@uni.minerva.edu', 'mounthees@keystone-foundation.org',
'jha.rs.rohit@gmail.com', 'prabhjeet_s.sias19@krea.ac.in', 'ayesha.wildlife@gmail.com', 'gururaja.kv@manipal.edu',
'sonalteotia89@gmail.com', 'vandana@keystone-foundation.org', 'chiranjibbora9@gmail.com', 'abhinavabkn10@gmail.com',
'abhishaek@ncf-india.org','vijayk@ncf-india.org', 'abdusshakur.mohammad@gmail.com', 'nishant.bhagwat@gmail.com',
'meghana.srivathsa@gmail.com', 'ant64@cornell.edu', 'vr292@cornell.edu', 'pkvenkadath@gmail.com',
'nidhichandkp345@gmail.com', 'cimilabob@gmail.com'
],
'type_B': [
'tasfiasyeda22@gmail.com', 'sameersingh.mail@gmail.com','navneet.singh@uni.minerva.edu','yana@uni.minerva.edu','ssangeethspeaks@gmail.com',
'ryansatish22@gmail.com','kvd.novel@gmail.com','sonalteotia89@gmail.com','ashankar@tifrh.res.in',
'vr292@cornell.edu','harshakumar@tifrh.res.in','cswathi@tifrh.res.in','chiranjibbora9@gmail.com',
'malavikamadhavan98@gmail.com','iqbalbhalla@gmail.com','vaishnav.rd@gmail.com','nibedita@naturalist.school',
'aarini.ghosh_phd18@ashoka.edu.in', 'navaneethsinigeorge2004@gmail.com','pradeeshwarrj@gmail.com','ramitsingal@gmail.com',
'Bioexplorer.ash@gmail.com','pravarmourya@gmail.com','poojamchoksi@gmail.com','sakhasiddha@gmail.com'
],
'background':[
'abdusshakur.mohammad@gmail.com', 'ankita.dasgupta@darukaa.com','chiranjibbora9@gmail.com',
'cimilabob@gmail.com', 'gaurav.singh@darukaa.com', 'gururaja.kv@manipal.edu', 'jha.rs.rohit@gmail.com',
'kvd.novel@gmail.com','malavikamadhavan98@gmail.com', 'mounthees@keystone-foundation.org',
'navneet.singh@uni.minerva.edu', 'nidhichandkp345@gmail.com','nishant.bhagwat@gmail.com','paulvpop@gmail.com',
'pkvenkadath@gmail.com','prabhjeet_s.sias19@krea.ac.in','rmathesan@keystone-foundation.org','sameersingh.mail@gmail.com',
'ssangeethspeaks@gmail.com','tasfiasyeda22@gmail.com','vandana@keystone-foundation.org','vr292@cornell.edu','yana@uni.minerva.edu']
}

metadata_template = {
    "metadata": {
        "title": "A large-scale crowd-sourced annotated acoustic dataset of Indian fauna: :dataType data",
        "upload_type": "publication",
        "publication_type": "article",
        "creators": zenodo_creator_list,
        "contributors": contributor_list,
        "access_right": "open",
        "license": "cc-by-nc-4.0"
    }
}



# 1. Configuration
ACCESS_TOKEN = cfg.zenodo_token
dep_ids = {"type_A": "18743214", "type_B": "18927866", "background": "18928201"}

FLAG_UPLOAD_ZIPS = False  # Make True if with this cell you want to upload zips to zenodo record
FLAG_UPDATE_META = True   # Make True if with this cell you want to update metadata of zeondo records


# 2. Get the Bucket URL for the deposition
def get_bucket_url(depo_id, token):
    deposition_url = f"https://zenodo.org/api/deposit/depositions/{depo_id}"
    r = requests.get(deposition_url, params={'access_token': token})
    r.raise_for_status()
    return r.json()['links']['bucket'], deposition_url

# 3. Upload a file
def upload_to_zenodo(file_path, bucket_url, token, execute=False):
    if not execute: return
    filename = os.path.basename(file_path)
    target_url = f"{bucket_url}/{filename}"

    print(f"Uploading {filename}...")
    with open(file_path, "rb") as fp:
        r = requests.put(
            target_url,
            data=fp,
            params={'access_token': token}
        )
    if r.status_code == 201 or r.status_code == 200:
        print(f"Successfully uploaded: {filename}")
    else:
        print(f"Failed to upload {filename}: {r.json()}")

def update_metadata(metadata, url, token, execute=False):
    if not execute: return
    response = requests.put(
        url,
        params={'access_token': ACCESS_TOKEN},
        data=json.dumps(metadata),
        headers={"Content-Type": "application/json"}
    )

    if response.status_code == 200:
        print("Metadata updated successfully!")
    else:
        print(f"Failed to update metadata: {response.status_code}")
        print(response.json())

# 4. Get Author details
def get_author_details(url):
    auth.authenticate_user()
    creds, _ = default()
    gc = gspread.authorize(creds)


    sh = gc.open_by_url(URL)
    worksheet = sh.get_worksheet(0) # or use sh.worksheet('SheetName')
    rows = worksheet.get_all_values()
    df = pd.DataFrame(rows[1:60], columns=rows[0])
    return df

def get_filtered_contributor_details(contributor_df, filter_list):
    filtered_df = contributor_df[contributor_df['Email ID'].isin(filter_list)].copy()
    filtered_df['name'] = filtered_df['Last Name'] + ', ' + filtered_df['First Name']
    filtered_df['affiliation'] = filtered_df['Primary Affiliation']
    filtered_df['orcid'] = filtered_df['ORCID (Optional)'].str.replace('https://orcid.org/', '').replace('http://orcid.org/', '')
    filtered_df['type'] = 'DataCollector'
    filtered_df = filtered_df[['name', 'affiliation', 'orcid', 'type']]
    return filtered_df.to_dict(orient='records')


# 4. Execution
author_df = get_author_details(AUTHOR_DETAILS_GSHEET_URL)

for dataType, DEPOSITION_ID in dep_ids.items():
    try:
        zip_folder_path = os.path.join(cfg.dataset_base_path, dataType)
        bucket, deposition_url = get_bucket_url(DEPOSITION_ID, ACCESS_TOKEN)

        # Iterate through zip files in the drive folder
        for file in os.listdir(zip_folder_path):
            if file.endswith(".zip"):
                full_path = os.path.join(zip_folder_path, file)
                upload_to_zenodo(full_path, bucket, ACCESS_TOKEN, FLAG_UPLOAD_ZIPS)
        contributor_details = get_filtered_contributor_details(author_df, contributor_list[dataType])
        record_metadata = metadata_template.copy()
        print(metadata_template)
        record_metadata['metadata']['title'] = metadata_template['metadata']['title'].replace(':dataType', dataType)
        record_metadata['metadata']['contributors'] = contributor_details
        print(record_metadata)
        update_metadata(record_metadata, deposition_url, ACCESS_TOKEN, FLAG_UPDATE_META)

    except Exception as e:
        print(f"An error occurred: {e}")
        traceback.print_exc()

{'metadata': {'title': 'A large-scale crowd-sourced annotated acoustic dataset of Indian fauna: :dataType data', 'upload_type': 'publication', 'publication_type': 'article', 'creators': [{'name': 'Singh, Sameer', 'affiliation': 'Independant', 'orcid': '0009-0000-7236-6042', 'type': 'DataManager'}, {'name': 'Ramesh, Vijay', 'affiliation': 'Cornell Lab of Ornithology', 'orcid': '0000-0002-0738-8808', 'type': 'ProjectLeader'}, {'name': 'Pop, Paul', 'affiliation': 'Ashoka Trust for Research in Ecology and the Environment', 'orcid': '0000-0003-0679-7789', 'type': 'DataCurator'}], 'contributors': {'type_A': ['priyankarchakraborty1991@gmail.com', 'sameersingh.mail@gmail.com', 'ramitsingal@gmail.com', 'ankita.dasgupta@darukaa.com', 'chandran@unknown.com', 'vishnu@keystone-foundation.org', 'anish@unknown.com', 'ashankar@tifrh.res.in', 'cswathi@tifrh.res.in', 'harshakumar@tifrh.res.in', 'kvd.novel@gmail.com', 'ssangeethspeaks@gmail.com', 'ryansatish22@gmail.com', 'komal.meena@darukaa.com', 'gaur